**Objective**

Clean and validate the DMS metadata generated in Notebook 01.

Input: data/processed/dms_metadata.csv

Outputs

data/processed/dms_cleaned.csv
reports/data_cleaning_log.csv

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROJECT_DIR = Path("/workspaces/DBA-ai-trust-score-framework-Interim")

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

REPORT_DIR = PROJECT_DIR / "reports"

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
metadata = pd.read_csv(
    PROCESSED_DIR / "dms_metadata.csv"
)

metadata.head()

,file_name,label,dataset,original_path
0,gB_7_s2_2019-03-11T14-12-25-01-00_ir_face_mp4-...,DangerousDriving,Thesis_dataset2,../data/raw/Thesis_dataset2/DangerousDriving/g...
1,gA_4_s2_ir_face_mp4-372_jpg.rf.448fec2e90c845f...,DangerousDriving,Thesis_dataset2,../data/raw/Thesis_dataset2/DangerousDriving/g...
2,gA_5_s2_ir_face_mp4-130_jpg.rf.1081aec5f8fd8da...,DangerousDriving,Thesis_dataset2,../data/raw/Thesis_dataset2/DangerousDriving/g...
3,gB_9_s2_2019-03-07T16-21-20-01-00_ir_face_mp4-...,DangerousDriving,Thesis_dataset2,../data/raw/Thesis_dataset2/DangerousDriving/g...
4,gA_5_s2_ir_face_mp4-246_jpg.rf.f7bcf481f361e68...,DangerousDriving,Thesis_dataset2,../data/raw/Thesis_dataset2/DangerousDriving/g...


In [4]:
print("Shape :", metadata.shape)

print("\nColumns")
print(metadata.columns.tolist())

print("\nData Types")
print(metadata.dtypes)

Shape : (1400, 4)

Columns
['file_name', 'label', 'dataset', 'original_path']

Data Types
file_name        str
label            str
dataset          str
original_path    str
dtype: object


______________________ Missing Values ___________________________

In [5]:
missing = metadata.isnull().sum()

missing = missing[
    missing > 0
].sort_values(ascending=False)

missing

Series([], dtype: int64)

______________________ Duplicate Values ___________________________

In [6]:
duplicates = metadata.duplicated().sum()

print("Duplicate Records :", duplicates)

Duplicate Records : 0


In [7]:
metadata.columns = (
    metadata.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

metadata.columns

Index(['file_name', 'label', 'dataset', 'original_path'], dtype='str')

In [8]:
label_map = {

    "closed":"ClosedEye",
    "Openeye":"OpenEye",
    "DangerousDriving":"DangerousDriving",
    "Distracted":"Distracted",
    "Drinking":"Drinking",
    "SafeDriving":"SafeDriving",
    "Yawn":"Yawn"

}

metadata["label"] = metadata["label"].replace(label_map)

metadata["label"].value_counts()

label
DangerousDriving    200
Distracted          200
Drinking            200
OpenEye             200
SafeDriving         200
Yawn                200
ClosedEye           200
Name: count, dtype: int64

In [9]:
metadata = metadata.drop_duplicates()

print(metadata.shape)

(1400, 4)


In [10]:
metadata = metadata.dropna(
    subset=["file_name"]
)

print(metadata.shape)

(1400, 4)


In [11]:
from pathlib import Path

missing_files = []

for path in metadata["original_path"]:

    if not Path(path).exists():
        missing_files.append(path)

print("Missing Image Files :", len(missing_files))

Missing Image Files : 0


____________________________ Data Cleaning Log ________________________________

In [12]:
cleaning_log = pd.DataFrame({

    "Issue":[

        "Missing Values",

        "Duplicate Records",

        "Missing Image Files"

    ],

    "Count":[

        metadata.isnull().sum().sum(),

        duplicates,

        len(missing_files)

    ],

    "Treatment":[

        "Removed missing file names",

        "Removed duplicates",

        "Validated paths"

    ]

})

cleaning_log

,Issue,Count,Treatment
0,Missing Values,0,Removed missing file names
1,Duplicate Records,0,Removed duplicates
2,Missing Image Files,0,Validated paths


In [13]:
metadata.to_csv(

    PROCESSED_DIR / "dms_cleaned.csv",

    index=False

)

cleaning_log.to_csv(

    REPORT_DIR / "data_cleaning_log.csv",

    index=False

)

print("✅ dms_cleaned.csv saved")

print("✅ data_cleaning_log.csv saved")

✅ dms_cleaned.csv saved
✅ data_cleaning_log.csv saved


In [14]:
print("Final Shape :", metadata.shape)

print("\nMissing Values")

print(
    metadata.isnull().sum().sum()
)

print("\nDuplicate Records")

print(
    metadata.duplicated().sum()
)

print("\nBehaviour Classes")

print(
    metadata["label"].value_counts()
)

Final Shape : (1400, 4)

Missing Values
0

Duplicate Records
0

Behaviour Classes
label
DangerousDriving    200
Distracted          200
Drinking            200
OpenEye             200
SafeDriving         200
Yawn                200
ClosedEye           200
Name: count, dtype: int64
